> Adapted from *Dive into Deep Learning*, section [Concise Implementation of Linear Regression](https://d2l.ai/chapter_linear-regression/linear-regression-concise.html). Not affiliated with or endorsed by the book's authors.
> Source, authors, licenses and changes: [Attribution and changes](#Attribution-and-changes), at the end of this notebook.

Previous: [Section 3.4: Linear Regression Implementation from Scratch](03.04_linear-regression-scratch.ipynb) | Next: [Section 3.6: Generalization](03.06_generalization.ipynb)

# Concise Implementation of Linear Regression

Deep learning has witnessed a sort of Cambrian explosion
over the past decade.
The sheer number of techniques, applications and algorithms by far surpasses the
progress of previous decades. 
This is due to a fortuitous combination of multiple factors,
one of which is the powerful free tools
offered by a number of open-source deep learning frameworks.
Theano (Bergstra et al., 2010),
DistBelief (Dean et al., 2012),
and Caffe (Jia et al., 2014)
arguably represent the
first generation of such models 
that found widespread adoption.
In contrast to earlier (seminal) works like
SN2 (Simulateur Neuristique) (Bottou and Le Cun, 1988),
which provided a Lisp-like programming experience,
modern frameworks offer automatic differentiation
and the convenience of Python.
These frameworks allow us to automate and modularize
the repetitive work of implementing gradient-based learning algorithms.

In [Section 3.4](https://d2l.ai/chapter_linear-regression/linear-regression-scratch.html#sec-linear-scratch), we relied only on
(i) tensors for data storage and linear algebra;
and (ii) automatic differentiation for calculating gradients.
In practice, because data iterators, loss functions, optimizers,
and neural network layers
are so common, modern libraries implement these components for us as well.
In this section, we will show you how to implement
the linear regression model from [Section 3.4](https://d2l.ai/chapter_linear-regression/linear-regression-scratch.html#sec-linear-scratch)
concisely by using high-level APIs of deep learning frameworks.

In [ ]:
import numpy as np

import divelet_ml as d2l
from divelet_ml import nn, optim

## Defining the Model

When we implemented linear regression from scratch
in [Section 3.4](https://d2l.ai/chapter_linear-regression/linear-regression-scratch.html#sec-linear-scratch),
we defined our model parameters explicitly
and coded up the calculations to produce output
using basic linear algebra operations.
You *should* know how to do this.
But once your models get more complex,
and once you have to do this nearly every day,
you will be glad of the assistance.
The situation is similar to coding up your own blog from scratch.
Doing it once or twice is rewarding and instructive,
but you would be a lousy web developer
if you spent a month reinventing the wheel.

For standard operations,
we can use a framework's predefined layers,
which allow us to focus
on the layers used to construct the model
rather than worrying about their implementation.
Recall the architecture of a single-layer network
as described in [Fig. 3.1.2](https://d2l.ai/chapter_linear-regression/linear-regression.html#fig-single-neuron).
The layer is called *fully connected*,
since each of its inputs is connected
to each of its outputs
by means of a matrix--vector multiplication.

In PyTorch, the fully connected layer is defined in `Linear` and `LazyLinear` classes (available since version 1.8.0). 
The latter
allows users to specify *merely*
the output dimension,
while the former
additionally asks for
how many inputs go into this layer.
Specifying input shapes is inconvenient and may require nontrivial calculations
(such as in convolutional layers).
Thus, for simplicity, we will use such "lazy" layers
whenever we can.
<img src="../tablet.svg" alt="divelet-ml" title="Added by divelet-ml, not in the book"> Here, `nn` is divelet-ml's NumPy stand-in for PyTorch's `torch.nn`; it provides `LazyLinear`.
A lazy layer has no weight before its first input,
so its `weight_init` and `bias_init` arguments set the initial values,
like the initializers of the JAX edition's `nn.Dense`.

In [ ]:
class LinearRegression(d2l.Module):  #@save
    """The linear regression model implemented with high-level APIs."""
    def __init__(self, lr):
        super().__init__()
        self.save_hyperparameters()
        self.net = nn.LazyLinear(1, weight_init=lambda shape: np.random.normal(0, 0.01, shape),
                                 bias_init=np.zeros)

In the `forward` method we just invoke the built-in `__call__` method of the predefined layers to compute the outputs.

In [ ]:
@d2l.add_to_class(LinearRegression)  #@save
def forward(self, X):
    return self.net(X)

## Defining the Loss Function

The `MSELoss` class computes the mean squared error (without the $1/2$ factor in [(3.1.5)](https://d2l.ai/chapter_linear-regression/linear-regression.html#equation-eq-mse)).
By default, `MSELoss` returns the average loss over examples.
It is faster (and easier to use) than implementing our own.

In [ ]:
@d2l.add_to_class(LinearRegression)  #@save
def loss(self, y_hat, y):
    fn = nn.MSELoss()
    return fn(y_hat, y)

## Defining the Optimization Algorithm

Minibatch SGD is a standard tool
for optimizing neural networks
and thus PyTorch supports it alongside a number of
variations on this algorithm in the `optim` module
<img src="../tablet.svg" alt="divelet-ml" title="Added by divelet-ml, not in the book"> (here, divelet-ml's `optim`, whose `SGD` takes the same arguments).
When we instantiate an `SGD` instance,
we specify the parameters to optimize over,
obtainable from our model via `self.parameters()`,
and the learning rate (`self.lr`)
required by our optimization algorithm.

In [ ]:
@d2l.add_to_class(LinearRegression)  #@save
def configure_optimizers(self):
    return optim.SGD(self.parameters(), self.lr)

## Training

You might have noticed that expressing our model through
high-level APIs of a deep learning framework
requires fewer lines of code.
We did not have to allocate parameters individually,
define our loss function, or implement minibatch SGD.
Once we start working with much more complex models,
the advantages of the high-level API will grow considerably.

Now that we have all the basic pieces in place,
the training loop itself is the same
as the one we implemented from scratch.
So we just call the `fit` method (introduced in [Section 3.2.4](https://d2l.ai/chapter_linear-regression/oo-design.html#oo-design-training)),
which relies on the implementation of the `fit_epoch` method
in [Section 3.4](https://d2l.ai/chapter_linear-regression/linear-regression-scratch.html#sec-linear-scratch),
to train our model.

In [ ]:
model = LinearRegression(lr=0.03)
data = d2l.SyntheticRegressionData(w=np.array([2, -3.4]), b=4.2)
trainer = d2l.Trainer(max_epochs=3)
trainer.fit(model, data)

Below, we
compare the model parameters learned
by training on finite data
and the actual parameters
that generated our dataset.
To access parameters,
we access the weights and bias
of the layer that we need.
As in our implementation from scratch,
note that our estimated parameters
are close to their true counterparts.

In [ ]:
@d2l.add_to_class(LinearRegression)  #@save
def get_w_b(self):
    return (self.net.weight, self.net.bias)

w, b = model.get_w_b()

In [ ]:
print(f'error in estimating w: {data.w - w.reshape(data.w.shape)}')
print(f'error in estimating b: {data.b - b}')

## Summary

This section contains the first
implementation of a deep network (in this book)
to tap into the conveniences afforded
by modern deep learning frameworks,
such as MXNet (Chen et al., 2015), 
JAX (Frostig et al., 2018), 
PyTorch (Paszke et al., 2019), 
and Tensorflow (Abadi et al., 2016).
We used framework defaults for loading data, defining a layer,
a loss function, an optimizer and a training loop.
Whenever the framework provides all necessary features,
it is generally a good idea to use them,
since the library implementations of these components
tend to be heavily optimized for performance
and properly tested for reliability.
At the same time, try not to forget
that these modules *can* be implemented directly.
This is especially important for aspiring researchers
who wish to live on the leading edge of model development,
where you will be inventing new components
that cannot possibly exist in any current library.

In PyTorch, the `data` module provides tools for data processing,
the `nn` module defines a large number of neural network layers and common loss functions.
We can initialize the parameters by replacing their values
with methods ending with `_`.
<img src="../tablet.svg" alt="divelet-ml" title="Added by divelet-ml, not in the book"> In divelet-ml, `d2l.ArrayLoader` takes the place of the `data` module,
`nn` and `optim` provide the few layers, losses and optimizers that adapted sections use,
and `LazyLinear`'s `weight_init` and `bias_init` arguments set the initial parameters.
Note that we need to specify the input dimensions of the network.
While this is trivial for now, it can have significant knock-on effects
when we want to design complex networks with many layers.
Careful considerations of how to parametrize these networks
is needed to allow portability.

## Exercises

1. How would you need to change the learning rate if you replace the aggregate loss over the minibatch
   with an average over the loss on the minibatch?
1. Review the framework documentation to see which loss functions are provided. In particular,
   replace the squared loss with Huber's robust loss function. That is, use the loss function
   $$l(y,y') = \begin{cases}|y-y'| -\frac{\sigma}{2} & \textrm{ if } |y-y'| > \sigma \\ \frac{1}{2 \sigma} (y-y')^2 & \textrm{ otherwise}\end{cases} \tag{3.5.1}$$
1. How do you access the gradient of the weights of the model? (Here, NumPy arrays do not store gradients: see `d2l.param_grads`.)
1. What is the effect on the solution if you change the learning rate and the number of epochs? Does it keep on improving?
1. How does the solution change as you vary the amount of data generated?
    1. Plot the estimation error for $\hat{\mathbf{w}} - \mathbf{w}$ and $\hat{b} - b$ as a function of the amount of data. Hint: increase the amount of data logarithmically rather than linearly, i.e., 5, 10, 20, 50, ..., 10,000 rather than 1000, 2000, ..., 10,000.
    2. Why is the suggestion in the hint appropriate?

Discussions of the original section: [book's forum](https://discuss.d2l.ai/t/45)

Previous: [Section 3.4: Linear Regression Implementation from Scratch](03.04_linear-regression-scratch.ipynb) | Next: [Section 3.6: Generalization](03.06_generalization.ipynb)

## Attribution and changes

Adapted from *Dive into Deep Learning*, section
[Concise Implementation of Linear Regression](https://d2l.ai/chapter_linear-regression/linear-regression-concise.html)
(source: [`chapter_linear-regression/linear-regression-concise.md`](https://github.com/d2l-ai/d2l-en/blob/23d7a5aecceee57d1292c56e90cce307f183bb0a/chapter_linear-regression/linear-regression-concise.md)
at commit `23d7a5aeccee`), by Aston Zhang, Zachary C. Lipton, Mu Li, Alexander J. Smola and contributors.
Text: [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). Code: adapted from the book's sample code, MIT-0.
Not affiliated with or endorsed by the book's authors.

**Changes from the original** (PyTorch edition):
- Code uses NumPy instead of PyTorch, so it runs in JupyterLite (Pyodide and xeus-python). The word *tensor* is kept and means a NumPy `ndarray`.
- `d2l` is divelet-ml's package (`import divelet_ml as d2l`) instead of the book's `d2l.torch`. `from torch import nn` becomes `from divelet_ml import nn, optim`: `nn.LazyLinear`, `nn.MSELoss` and `optim.SGD` are NumPy stand-ins for PyTorch's, with the same arguments. The prose says so.
- `self.net.weight.data.normal_(0, 0.01)` and `self.net.bias.data.fill_(0)` become `LazyLinear`'s `weight_init` and `bias_init` arguments: a lazy layer has no weight until its first input, and NumPy arrays have no in-place initializers. `Trainer.fit` runs the model once before `configure_optimizers`, so `self.parameters()` finds the weight and bias.
- NumPy has no automatic differentiation, so gradients come from [autograd](https://github.com/HIPS/autograd), as in [2.5](https://d2l.ai/chapter_preliminaries/autograd.html): `d2l.param_grads(fn, model, params)` returns the gradients of `fn()` with respect to the model's parameter arrays, in place of PyTorch's `loss.backward()` and `param.grad`. The trainer's `fit_epoch` uses it, and its optimizer's `step(grads)` updates the parameters in place.
- `torch.optim.SGD(self.parameters(), self.lr)` becomes `optim.SGD(self.parameters(), self.lr)`. `get_w_b` returns the arrays themselves (there is no `.data`), and `d2l.tensor` and `d2l.reshape` become `np.array` and the array's `reshape`.
- Exercise 3 notes that gradients are not stored on the parameters.
- As in the original, the data and the initial weights are random and differ on every run.
- The d2lbook framework tabs for MXNet, TensorFlow and JAX are removed. Cross-references to other sections link to the book's site, and citations are written inline (author and year); the book's bibliography lists them.